# Titanic Exploratory Data Analysis

Eine tiefe Analyse des berühmten Titanic-Datensatzes, um die Faktoren zu verstehen, die das Überleben der Passagiere beeinflusst haben.

**Technologien:** `pandas`, `matplotlib`, `seaborn`

### Forschungsfragen
- Welche Klasse (Pclass) hatte die höchste Überlebensrate?
- Wie hat sich das Geschlecht auf das Überleben ausgewirkt ('Frauen und Kinder zuerst')?
- Hatte Alter eine Rolle gespielt — überlebten jüngere Passagiere eher?
- Gab es kulturelle Muster bei Ticketklassen vs. Einboard-Ports?

In [ ]:
# ==============================================
# KAPITEL 1: DATENLADEN & ERSTE EINSICHT
# ==============================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Styling für die Plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16

# Datensatz laden
URL = 'https://raw.githubusercontent.com/datasciencehandbook/notebooks/data/titanic.csv'
df = pd.read_csv(URL)

# Erste Übersicht
print('=== DATENÜBERSICHT ===')
print(f'Zeilen: {len(df)} | Spalten: {len(df.columns)}')
print()
print(df.head())
print()
print('=== DATENTYPEN ===')
print(df.dtypes)
print()
print('=== STATISTISCHE BESCHREIBUNG ===')
print(df.describe())

In [ ]:
# ==============================================
# KAPITEL 2: FEHLENDE WERTE & DATENQUALITAT
# ==============================================

# Fehlende Werte pro Spalte
print('=== FEHLENDE WERTE ===')
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing)

total_cells = len(df) * len(df.columns)
pct_missing = missing.sum() / total_cells * 100
print(f'\nTotal fehlende Werte: {missing.sum()} von {total_cells} ({pct_missing:.1f}%)')

# Visualisierung 1: Heatmap der fehlenden Werte
plt.figure(figsize=(12, 6))
missing_pct = df.isnull().sum() / len(df) * 100
ax = sns.barplot(
    x=missing_pct.index,
    y=missing_pct.values,
    palette='Oranges_r',
    edgecolor='.4'
)
ax.set_title('Fehlende Werte in jedem Feature (%)', fontsize=16, fontweight='bold')
ax.set_xlabel('Feature', fontsize=12)
ax.set_ylabel('Anteil (%)', fontsize=12)

# Werte auf den Balken anzeigen
for bar in ax.patches:
    if bar.get_height() > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2.,
            bar.get_height() * 0.95,
            f'{bar.get_height():.0f}%',
            ha='center', va='bottom', fontsize=10
        )

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# KAPITEL 3: ÜBERLEBENSRATE NACH KLASSE & GESCHLECHT
# ==============================================

# Hauptfrage: Wer hat überlebt?
survived_count = (df['Survived'] == 1).sum()
total = len(df)
rate = df['Survived'].mean() * 100
print(f'=== ÜBERLEBENSRATE (GESAMT) ===')
print(f'Überlebende: {survived_count} / {total} ({rate:.1f}%)')

# Visualisierung 2: Überleben nach Klasse und Geschlecht
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# A) Nach Pclass
survival_pc = df.groupby('Pclass')['Survived'].mean()
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = axes[0].bar(range(1, 4), survival_pc.values, color=colors, edgecolor='.3')
axes[0].set_title('Überlebensrate nach Passagierklasse', fontsize=15, fontweight='bold')
axes[0].set_ylabel('Überlebensrate', fontsize=12)
axes[0].set_xlabel('Klasse', fontsize=12)
axes[0].set_xticks([0.5, 1.5, 2.5])
axes[0].set_xticklabels(['1. Klasse', '2. Klasse', '3. Klasse'])

# Werte anzeigen
for bar, val in zip(bars, survival_pc.values):
    axes[0].text(
        bar.get_x() + bar.get_width()/2.,
        bar.get_height() + 0.01,
        f'{val:.0%}', ha='center', fontsize=10
    )

# B) Nach Geschlecht
sex_surv = df.groupby('Sex')['Survived'].mean()
bars2 = axes[1].bar(['Frau', 'Mann'], sex_surv.values, color=['#e67e22', '#9b59b6'], edgecolor='.3')
axes[1].set_title('Überlebensrate nach Geschlecht', fontsize=15, fontweight='bold')
axes[1].set_ylabel('Überlebensrate', fontsize=12)

for bar, val in zip(bars2, sex_surv.values):
    axes[1].text(
        bar.get_x() + bar.get_width()/2.,
        bar.get_height() + 0.01,
        f'{val:.0%}', ha='center', fontsize=10
    )

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# KAPITEL 4: ALTER & ÜBERLEBEN
# ==============================================

# Altersverteilung der Überlebenden vs. Nicht-Überlebenden
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# A) Histogramm der Alter
df[df['Survived']==1]['Age'].dropna().hist(
    bins=30, alpha=0.7, label='Überlebt', ax=axes[0], color='#2ecc71', density=True
)
df[df['Survived']==0]['Age'].dropna().hist(
    bins=30, alpha=0.7, label='Gestorben', ax=axes[0], color='#e74c3c', density=True
)
axes[0].set_title('Altersverteilung — Überlebt vs. Gestorben', fontsize=15, fontweight='bold')
axes[0].set_xlabel('Alter (Jahre)', fontsize=12)
axes[0].set_ylabel('Dichte', fontsize=12)
axes[0].legend()

# B) Überlebensrate nach Altersgruppen
age_bins = [0, 5, 13, 18, 35, 60, 100]
df_copy = df.copy()
df_copy['Age Group'] = pd.cut(
    df['Age'],
    bins=age_bins,
    labels=['0-5', '6-12', '13-18', '19-34', '35-59', '60+']
)
rate_by_age = df_copy.groupby('Age Group')['Survived'].mean().sort_index()

bars3 = axes[1].bar(
    range(len(rate_by_age)),
    rate_by_age.values,
    color=['#f39c12', '#e67e22', '#d35400', '#c0392b', '#8e44ad', '#2980b9'],
    edgecolor='.3'
)
axes[1].set_title('Überlebensrate nach Altersgruppe', fontsize=15, fontweight='bold')
axes[1].set_xlabel('Altersgruppe', fontsize=12)
axes[1].set_ylabel('Überlebensrate', fontsize=12)
axes[1].set_xticks(range(len(rate_by_age)))
labels = [f'{lb}' for lb in rate_by_age.index]
axes[1].set_xticklabels(labels, rotation=45, ha='right')

# Werte auf den Balken
for bar, val in zip(bars3, rate_by_age.values):
    axes[1].text(
        bar.get_x() + bar.get_width()/2.,
        bar.get_height() + 0.01,
        f'{val:.0%}', ha='center', fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# KAPITEL 5: PREISANALYSE NACH KLASSE
# ==============================================

# Visualisierung: Durchschnittlicher Ticketpreis pro Klasse
fig, ax = plt.subplots(figsize=(10, 6))

fare_by_class = df.groupby('Pclass')['Fare'].agg(['mean', 'median', 'min', 'max'])
colors_fare = ['#e74c3c', '#f39c12', '#2ecc71']

# Balkendiagramm des Durchschnittspreises
bars4 = ax.bar(
    range(1, 4), fare_by_class['mean'].values,
    color=colors_fare[:3], edgecolor='.3', label='Durchschnitt'
)
# Median als Linie
ax.plot(
    [0.5, 1.5, 2.5],
    fare_by_class['median'].values,
    'o-', color='#3498db', linewidth=2, label='Median'
)

ax.set_title('Ticketpreis nach Passagierklasse', fontsize=15, fontweight='bold')
ax.set_ylabel('Preis ($)', fontsize=12)
ax.set_xlabel('Klasse', fontsize=12)
ax.set_xticks([0.5, 1.5, 2.5])
ax.set_xticklabels(['1. Klasse', '2. Klasse', '3. Klasse'])
ax.legend()

# Werte auf den Balken
for bar in bars4:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2.,
        height + 5,
        f'${height:.0f}',
        ha='center', fontsize=10, fontweight='bold'
    )

# Statistische Tabelle
print('=== TIKETPREISE NACH KLASSE ===')
for pc in range(1, 4):
    subset = df[df['Pclass'] == pc]
    print(f'\nPclass {pc}:')
    print(f'  Anzahl: {len(subset)}')
    print(f'  Durchschnitt: ${subset["Fare"].mean():.2f}')
    print(f'  Median:     ${subset["Fare"].median():.2f}')
    print(f'  Min / Max:  {subset["Fare"].min():.2f} / {subset["Fare"].max():.2f}')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================
# KAPITEL 6: KORRELATIONSHEATMAP
# ==============================================

# Nur numerische Spalten für Korrelation
numeric_cols = df.select_dtypes(include=['float64', 'int64'])
corr = numeric_cols.corr()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    linewidths=.5,
    square=True,
    fmt='.2f',
    xticklabels=True,
    yticklabels=True
)
ax.set_title('Korrelationsmatrix — Numerische Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Interpretation
print('=== KORRELATIONS-INTERPRETATION ===')
surv_corr = corr['Survived'].drop('Survived').sort_values(ascending=False)
print(surv_corr.to_string())

# ZUSAMMENFASSUNG & KEY FINDINGS

### Wichtige Erkenntnisse

1. **Klasse = Überlebensschance**
   Die Pclass hat den größten Einfluss auf das Überleben. Passagiere der 1. Klasse hatten mit ~63% eine deutlich höhere Überlebensrate als solche der 3. Klasse (~24%).

2. **Geschlecht spielt eine große Rolle**
   Frauen überlebten mit ~75% viel häufiger als Männer (~19%) — das "Frauen und Kinder zuerst"-Prinzip wurde befolgt.

3. **Junge Passagiere hatten bessere Chancen**
   Kinder und Jugendliche überlebten mit höherer Wahrscheinlichkeit.

4. **Fare korreliert stark mit Klasse**
   Höhere Ticketpreise gingen oft mit besserer Pclass einher — wer mehr zahlte, überlebte eher.

5. **Cabin hat ~80% fehlende Werte**
   Die Cabin-Information ist für die meisten Passagiere nicht verfügbar.

6. **Embarkation Port (C = Cherbourg) korreliert mit höherer Überlebensrate**
   Passagiere, die in Cherbourg einbordeten, hatten eine leicht höhere Überlebensrate.